# Shrimp Disease Auto-Augmentation Benchmark

Benchmarks the same TIMM/Torchvision lightweight classifiers with three separate auto-augmentation policies: TrivialAugmentWide, RandAugment, and AugMix. Validation/test transforms and the fixed image-level split are kept constant across runs.


In [ ]:
print("Running Kaggle auto-augmentation benchmark. Input data is read from /kaggle/input; outputs are written to /kaggle/working.")


## 1. Install dependencies



In [ ]:
import importlib.util
import subprocess
import sys

packages = [
    "timm>=1.0.0",
]


def import_name_for(package_spec: str) -> str:
    package = package_spec.split(">=")[0].split("==")[0].split("<")[0]
    return package.replace("-", "_")


missing = [pkg for pkg in packages if importlib.util.find_spec(import_name_for(pkg)) is None]
if missing:
    print("Installing missing dependencies:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *missing])
else:
    print("All training dependencies are already available.")


## 2. Configuration and fixed split



In [ ]:
import gc
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as tv_models
import torchvision.transforms as transforms
from PIL import Image
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import InterpolationMode
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

IMG_SIZE = 224
EPOCHS = 30
PATIENCE = 5
PAPER_BATCH_SIZE = 128
MICRO_BATCH_SIZE = 32
ACCUMULATION_STEPS = max(1, PAPER_BATCH_SIZE // MICRO_BATCH_SIZE)
EVAL_BATCH_SIZE = PAPER_BATCH_SIZE
WARMUP_EPOCHS = 5
WARMUP_HEAD_LR = 1e-3
BACKBONE_FINETUNE_LR = 2e-5
HEAD_FINETUNE_LR = 1e-4
STEP_SIZE = 3
STEP_GAMMA = 0.9
NUM_WORKERS = 2
ENABLE_GPU_LOGGING = True
GPU_LOG_EVERY_N_EPOCHS = 1

DATA_DIR = Path("/kaggle/input/datasets/uynnhy/processed-images/processed_images")
OUTPUT_DIR = Path("/kaggle/working/full_lightweight_model_autoaugment_comparison")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
REPORT_DIR = OUTPUT_DIR / "reports"

for directory in [CHECKPOINT_DIR, REPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

CLASS_DIRS = ["1. Healthy", "2. BG", "3. WSSV", "4. WSSV_BG"]
CLASS_NAMES = ["Healthy", "BG", "WSSV", "WSSV_BG"]
CLASS_TO_IDX = {folder: idx for idx, folder in enumerate(CLASS_DIRS)}
NUM_CLASSES = len(CLASS_DIRS)

TIMM_MODELS = [
    "convnext_tiny_in22k",
    "mobilenet_v3_large",
    "efficientnet_b0",
    "repvgg_a0",
    "efficientnet_v2_s",
    "shufflenet_v2_x1_0",
    "fastvit_t8",
    "edgenext_xx_small",
    "mobileone_s0",
    "mobilevit_s",
    "mobilenetv4_conv_small",
    "mnasnet_100",
    "ghostnetv2_100",
    "rexnet_100",
    "squeezenet1_1",
    "mobilenetv4_hybrid_medium",
    "efficientvit_m1",
]
ALL_MODELS = TIMM_MODELS.copy()

AUTO_AUGMENT_POLICIES = {
    "trivial_aug": "TrivialAugmentWide",
    "randaug": "RandAugment",
    "augmix": "AugMix",
}


def gpu_status_text() -> str:
    if not torch.cuda.is_available():
        return "CUDA unavailable"

    allocated = torch.cuda.memory_allocated() / 1024**2
    reserved = torch.cuda.memory_reserved() / 1024**2
    max_allocated = torch.cuda.max_memory_allocated() / 1024**2
    text = f"torch CUDA memory allocated/reserved/max: {allocated:.1f}/{reserved:.1f}/{max_allocated:.1f} MB"

    try:
        completed = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=utilization.gpu,memory.used,memory.total,power.draw,temperature.gpu",
                "--format=csv,noheader,nounits",
            ],
            capture_output=True,
            text=True,
            timeout=5,
        )
        if completed.returncode == 0:
            first_gpu = completed.stdout.strip().splitlines()[0]
            util, mem_used, mem_total, power, temp = [part.strip() for part in first_gpu.split(",")[:5]]
            text += f" | nvidia-smi util={util}% mem={mem_used}/{mem_total} MB power={power} W temp={temp} C"
        else:
            text += f" | nvidia-smi failed: {completed.stderr.strip()[-200:]}"
    except Exception as exc:
        text += f" | nvidia-smi unavailable: {type(exc).__name__}: {exc}"

    return text


def print_gpu_status(label: str):
    if ENABLE_GPU_LOGGING:
        print(f"[GPU] {label}: {gpu_status_text()}")


print_gpu_status("startup")



In [ ]:
def reset_random_state(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)
    torch.manual_seed(worker_seed)


def seeded_generator(seed: int = SEED):
    generator = torch.Generator()
    generator.manual_seed(seed)
    return generator


def discover_processed_images(data_dir: Path) -> pd.DataFrame:
    rows = []
    for class_dir in CLASS_DIRS:
        folder = data_dir / class_dir
        if not folder.exists():
            print(f"Warning: missing class folder: {folder}")
            continue
        for path in sorted(folder.iterdir()):
            if path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}:
                rows.append({"path": str(path), "class_dir": class_dir, "label": CLASS_TO_IDX[class_dir]})
    frame = pd.DataFrame(rows)
    if frame.empty:
        raise RuntimeError(f"No images found under {data_dir}. Run preprocessing from the baseline first.")
    return frame


df = discover_processed_images(DATA_DIR)
print(f"Loaded {len(df)} processed images from {DATA_DIR}")
display(df["class_dir"].value_counts().reindex(CLASS_DIRS).rename("count").to_frame())

train_df, tmp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=SEED,
    shuffle=True,
)
val_df, test_df = train_test_split(
    tmp_df,
    test_size=0.50,
    stratify=tmp_df["label"],
    random_state=SEED,
    shuffle=True,
)

for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{split_name}: {len(split_df)} images")
    print(split_df["class_dir"].value_counts().reindex(CLASS_DIRS).to_dict())

assert set(train_df["path"]).isdisjoint(set(val_df["path"]))
assert set(train_df["path"]).isdisjoint(set(test_df["path"]))
assert set(val_df["path"]).isdisjoint(set(test_df["path"]))
print("Image-level split overlap check passed.")



## 3. TIMM/Torchvision data pipeline and helpers


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def instantiate_transform(transform_cls, **kwargs):
    try:
        return transform_cls(**kwargs)
    except TypeError:
        kwargs.pop("interpolation", None)
        return transform_cls(**kwargs)


def build_auto_augment(augmentation_key: str):
    if augmentation_key == "trivial_aug":
        return instantiate_transform(transforms.TrivialAugmentWide, interpolation=InterpolationMode.BILINEAR)
    if augmentation_key == "randaug":
        return instantiate_transform(transforms.RandAugment, num_ops=2, magnitude=9, interpolation=InterpolationMode.BILINEAR)
    if augmentation_key == "augmix":
        return instantiate_transform(
            transforms.AugMix,
            severity=3,
            mixture_width=3,
            chain_depth=-1,
            alpha=1.0,
            interpolation=InterpolationMode.BILINEAR,
        )
    raise ValueError(f"Unsupported augmentation policy: {augmentation_key}")


def build_train_transform(augmentation_key: str):
    return transforms.Compose([
        transforms.Resize(256, interpolation=InterpolationMode.BICUBIC, antialias=True),
        transforms.RandomResizedCrop(
            IMG_SIZE,
            scale=(0.82, 1.0),
            ratio=(0.90, 1.10),
            interpolation=InterpolationMode.BICUBIC,
            antialias=True,
        ),
        build_auto_augment(augmentation_key),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


eval_transform = transforms.Compose([
    transforms.Resize(236, interpolation=InterpolationMode.BICUBIC, antialias=True),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class ShrimpFrameDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, transform=None):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, int(row["label"])


def make_train_loader(augmentation_key: str):
    return DataLoader(
        ShrimpFrameDataset(train_df, build_train_transform(augmentation_key)),
        batch_size=MICRO_BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        worker_init_fn=seed_worker,
        generator=seeded_generator(SEED),
    )


val_loader = DataLoader(
    ShrimpFrameDataset(val_df, eval_transform),
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    worker_init_fn=seed_worker,
    generator=seeded_generator(SEED),
)
test_loader = DataLoader(
    ShrimpFrameDataset(test_df, eval_transform),
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    worker_init_fn=seed_worker,
    generator=seeded_generator(SEED),
)


In [ ]:
TIMM_ALIASES = {
    "convnext_tiny_in22k": ["convnext_tiny.fb_in22k", "convnext_tiny.fb_in1k", "convnext_tiny"],
    "mobilenet_v3_large": ["mobilenetv3_large_100.ra_in1k", "mobilenetv3_large_100", "tf_mobilenetv3_large_100"],
    "efficientnet_b0": ["efficientnet_b0.ra_in1k", "tf_efficientnet_b0.ns_jft_in1k", "efficientnet_b0"],
    "repvgg_a0": ["repvgg_a0.rvgg_in1k", "repvgg_a0"],
    "efficientnet_v2_s": ["tf_efficientnetv2_s.in21k_ft_in1k", "tf_efficientnetv2_s", "efficientnetv2_rw_s.ra2_in1k"],
    "shufflenet_v2_x1_0": ["shufflenet_v2_x1_0"],
    "fastvit_t8": ["fastvit_t8.apple_in1k", "fastvit_t8"],
    "edgenext_xx_small": ["edgenext_xx_small.in1k", "edgenext_xx_small"],
    "mobileone_s0": ["mobileone_s0.apple_in1k", "mobileone_s0"],
    "mobilevit_s": ["mobilevit_s.cvnets_in1k", "mobilevit_s"],
    "mobilenetv4_conv_small": ["mobilenetv4_conv_small.e2400_r224_in1k", "mobilenetv4_conv_small"],
    "mnasnet_100": ["mnasnet_100.rmsp_in1k", "mnasnet_100"],
    "ghostnetv2_100": ["ghostnetv2_100.in1k", "ghostnetv2_100"],
    "rexnet_100": ["rexnet_100.nav_in1k", "rexnet_100"],
    "squeezenet1_1": ["squeezenet1_1"],
    "mobilenetv4_hybrid_medium": ["mobilenetv4_hybrid_medium.e200_r256_in12k_ft_in1k", "mobilenetv4_hybrid_medium"],
    "efficientvit_m1": ["efficientvit_m1.r224_in1k", "efficientvit_m1"],
}


def sanitize_name(name: str) -> str:
    return name.replace("/", "_").replace(" ", "_").replace(".", "_")


def count_params(model) -> float:
    return sum(param.numel() for param in model.parameters()) / 1e6


def resolve_timm_name(display_name: str) -> str:
    candidates = TIMM_ALIASES.get(display_name, [display_name])
    available = set(timm.list_models(pretrained=False))
    for candidate in candidates:
        if candidate in available:
            return candidate
    pattern_hits = []
    for candidate in candidates:
        pattern_hits.extend(timm.list_models(candidate + "*", pretrained=False))
    if pattern_hits:
        return sorted(pattern_hits)[0]
    raise ValueError(f"No TIMM model found for {display_name}. Tried: {candidates}")


class TimmShrimpXNet(nn.Module):
    def __init__(self, timm_name: str, pretrained: bool):
        super().__init__()
        try:
            self.backbone = timm.create_model(timm_name, pretrained=pretrained, num_classes=0, global_pool="avg")
        except TypeError:
            self.backbone = timm.create_model(timm_name, pretrained=pretrained, num_classes=0)

        # TIMM's num_features can describe the pre-classifier channel count,
        # while num_classes=0 can return a post-head embedding for some models
        # such as MobileNetV3. Infer the actual forward output to avoid head
        # shape mismatches.
        was_training = self.backbone.training
        self.backbone.eval()
        with torch.no_grad():
            sample = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
            features = self.backbone(sample)
            if isinstance(features, (list, tuple)):
                features = features[-1]
            num_features = features.flatten(1).shape[1]
        self.backbone.train(was_training)

        self.classifier = nn.Linear(num_features, NUM_CLASSES)

    def forward(self, x):
        x = self.backbone(x)
        if isinstance(x, (list, tuple)):
            x = x[-1]
        x = torch.flatten(x, 1)
        return self.classifier(x)


class TorchvisionShrimpXNet(nn.Module):
    def __init__(self, model_name: str, pretrained: bool):
        super().__init__()
        if model_name == "shufflenet_v2_x1_0":
            weights = tv_models.ShuffleNet_V2_X1_0_Weights.IMAGENET1K_V1 if pretrained else None
            backbone = tv_models.shufflenet_v2_x1_0(weights=weights)
            num_features = backbone.fc.in_features
            backbone.fc = nn.Identity()
        elif model_name == "squeezenet1_1":
            weights = tv_models.SqueezeNet1_1_Weights.IMAGENET1K_V1 if pretrained else None
            backbone = tv_models.squeezenet1_1(weights=weights)
            num_features = 512
            backbone.classifier = nn.AdaptiveAvgPool2d((1, 1))
        else:
            raise ValueError(f"Unsupported torchvision fallback model: {model_name}")

        self.backbone = backbone
        self.classifier = nn.Linear(num_features, NUM_CLASSES)

    def forward(self, x):
        x = self.backbone(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


def create_timm_classifier(display_name: str):
    if display_name in {"shufflenet_v2_x1_0", "squeezenet1_1"}:
        try:
            return TorchvisionShrimpXNet(display_name, pretrained=True).to(device), display_name, True
        except Exception as pretrained_error:
            print(
                f"Pretrained torchvision weights failed for {display_name}: "
                f"{type(pretrained_error).__name__}: {pretrained_error}"
            )
            return TorchvisionShrimpXNet(display_name, pretrained=False).to(device), display_name, False

    timm_name = resolve_timm_name(display_name)
    try:
        model = TimmShrimpXNet(timm_name, pretrained=True)
        pretrained = True
    except Exception as pretrained_error:
        print(
            f"Pretrained weights failed for {display_name} ({timm_name}): "
            f"{type(pretrained_error).__name__}: {pretrained_error}"
        )
        model = TimmShrimpXNet(timm_name, pretrained=False)
        pretrained = False
    return model.to(device), timm_name, pretrained


def classifier_parameters(model):
    if hasattr(model, "classifier"):
        return list(model.classifier.parameters())

    params = []
    classifier = model.get_classifier() if hasattr(model, "get_classifier") else None
    if isinstance(classifier, nn.Module):
        params = list(classifier.parameters())
    elif isinstance(classifier, (list, tuple, nn.ModuleList)):
        for module in classifier:
            if isinstance(module, nn.Module):
                params.extend(list(module.parameters()))

    if not params:
        head_tokens = ("classifier", "head", "fc")
        params = [
            param
            for name, param in model.named_parameters()
            if any(token in name.lower() for token in head_tokens)
        ]

    if not params:
        raise RuntimeError("Could not identify classifier/head parameters for warmup.")
    return params


def freeze_backbone_for_warmup(model):
    for param in model.parameters():
        param.requires_grad = False
    for param in classifier_parameters(model):
        param.requires_grad = True


def make_warmup_optimizer(model):
    return optim.Adam(
        [param for param in model.parameters() if param.requires_grad],
        lr=WARMUP_HEAD_LR,
    )


def unfreeze_module(module):
    for param in module.parameters():
        param.requires_grad = True


def unfreeze_final_backbone_portion(model, display_name: str):
    backbone = model.backbone if hasattr(model, "backbone") else model
    trainable_modules = []

    for param in model.parameters():
        param.requires_grad = False

    for param in classifier_parameters(model):
        param.requires_grad = True

    if hasattr(backbone, "features") and isinstance(backbone.features, (nn.Sequential, nn.ModuleList, list, tuple)):
        features = backbone.features
        if "convnext" in display_name.lower() and len(features) > 5:
            selected = list(features[5:])
        else:
            start = max(0, len(features) - max(1, len(features) // 3))
            selected = list(features[start:])
        trainable_modules.extend(selected)

    elif hasattr(backbone, "stages") and isinstance(backbone.stages, (nn.Sequential, nn.ModuleList, list, tuple)):
        stages = backbone.stages
        start = max(0, len(stages) - max(1, len(stages) // 3))
        trainable_modules.extend(list(stages[start:]))

    elif hasattr(backbone, "blocks") and isinstance(backbone.blocks, (nn.Sequential, nn.ModuleList, list, tuple)):
        blocks = backbone.blocks
        start = max(0, len(blocks) - max(1, len(blocks) // 3))
        trainable_modules.extend(list(blocks[start:]))

    else:
        excluded = {"classifier", "head", "fc", "global_pool", "pool", "avgpool"}
        children = [
            child for name, child in backbone.named_children()
            if name not in excluded and not name.startswith("head")
        ]
        trainable_modules.extend(children[-2:] if len(children) >= 2 else children)

    for attr in ["norm", "norm_head", "head_norm", "pre_head", "final_conv"]:
        module = getattr(backbone, attr, None)
        if isinstance(module, nn.Module):
            trainable_modules.append(module)

    for module in trainable_modules:
        unfreeze_module(module)

    return sum(param.numel() for param in model.parameters() if param.requires_grad)


def make_finetune_optimizer(model):
    head_param_ids = {id(param) for param in classifier_parameters(model)}
    backbone_params = []
    head_params = []
    for param in model.parameters():
        if not param.requires_grad:
            continue
        if id(param) in head_param_ids:
            head_params.append(param)
        else:
            backbone_params.append(param)

    param_groups = []
    if backbone_params:
        param_groups.append({"params": backbone_params, "lr": BACKBONE_FINETUNE_LR})
    if head_params:
        param_groups.append({"params": head_params, "lr": HEAD_FINETUNE_LR})
    return optim.Adam(param_groups)


def predict_pytorch(model, loader, criterion=None, timed=False):
    model.eval()
    total_loss = 0.0
    all_labels = []
    all_preds = []

    if timed:
        dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
        with torch.no_grad():
            for _ in range(5):
                model(dummy)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
        start = time.time()
    else:
        start = None

    with torch.no_grad():
        for ims, gts in loader:
            ims = ims.to(device, non_blocking=True)
            gts = gts.to(device, non_blocking=True)
            logits = model(ims)
            if criterion is not None:
                total_loss += criterion(logits, gts).item() * ims.size(0)
            preds = torch.argmax(logits, dim=1)
            all_labels.extend(gts.cpu().numpy().tolist())
            all_preds.extend(preds.cpu().numpy().tolist())

    if timed and torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.time() - start if timed else None

    return {
        "loss": total_loss / max(1, len(all_labels)) if criterion is not None else None,
        "accuracy": accuracy_score(all_labels, all_preds),
        "macro_f1": f1_score(all_labels, all_preds, average="macro", zero_division=0),
        "cohen_kappa": cohen_kappa_score(all_labels, all_preds),
        "elapsed": elapsed,
        "labels": all_labels,
        "preds": all_preds,
    }



## 4. Train TIMM models



In [ ]:
def train_timm_model(model_name: str, augmentation_key: str, train_loader: DataLoader) -> dict:
    print("\n" + "=" * 90)
    print(f"Training TIMM classifier: {model_name} | Augmentation: {AUTO_AUGMENT_POLICIES[augmentation_key]}")
    print("=" * 90)

    model, timm_name, pretrained = create_timm_classifier(model_name)
    print(f"Resolved TIMM model: {timm_name} | pretrained={pretrained}")
    print_gpu_status(f"{model_name} after model.to(device)")
    criterion = nn.CrossEntropyLoss()
    safe_name = sanitize_name(model_name)
    best_path = CHECKPOINT_DIR / f"best_{augmentation_key}_{safe_name}.pth"
    best_val_loss = float("inf")
    best_val_f1 = -1.0
    epochs_no_improve = 0
    train_start = time.time()

    freeze_backbone_for_warmup(model)
    print(f"Warmup trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    optimizer = make_warmup_optimizer(model)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=STEP_SIZE, gamma=STEP_GAMMA)

    for epoch in range(EPOCHS):
        if epoch == WARMUP_EPOCHS:
            print("\n--- Switching to fine-tuning phase: unfreezing backbone with lower LR ---")
            trainable_count = unfreeze_final_backbone_portion(model, model_name)
            print(f"Fine-tune trainable parameters: {trainable_count:,}")
            optimizer = make_finetune_optimizer(model)
            scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=STEP_SIZE, gamma=STEP_GAMMA)
            epochs_no_improve = 0

        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        current_lrs = [group["lr"] for group in optimizer.param_groups]
        optimizer.zero_grad(set_to_none=True)

        for step, (ims, gts) in enumerate(tqdm(train_loader, desc=f"{AUTO_AUGMENT_POLICIES[augmentation_key]} | {model_name} epoch {epoch + 1}/{EPOCHS}", leave=False)):
            ims = ims.to(device, non_blocking=True)
            gts = gts.to(device, non_blocking=True)

            logits = model(ims)
            loss = criterion(logits, gts)
            (loss / ACCUMULATION_STEPS).backward()

            if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader):
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)

            train_loss += loss.item() * ims.size(0)
            train_correct += (torch.argmax(logits, dim=1) == gts).sum().item()
            train_total += gts.size(0)

        scheduler.step()
        val_metrics = predict_pytorch(model, val_loader, criterion=criterion, timed=False)
        if (epoch + 1) % GPU_LOG_EVERY_N_EPOCHS == 0:
            print_gpu_status(f"{model_name} epoch {epoch + 1} end")
        train_acc = train_correct / max(1, train_total)
        phase = "warmup" if epoch < WARMUP_EPOCHS else "finetune"
        lr_text = ",".join(f"{lr:.2e}" for lr in current_lrs)
        print(
            f"Epoch {epoch + 1:02d}/{EPOCHS} | "
            f"Phase: {phase} | LR: {lr_text} | "
            f"Train Loss: {train_loss / max(1, train_total):.4f} - Acc: {train_acc:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} - Acc: {val_metrics['accuracy']:.4f} - Macro F1: {val_metrics['macro_f1']:.4f}"
        )

        improved = val_metrics["loss"] < best_val_loss
        if improved:
            best_val_loss = val_metrics["loss"]
            best_val_f1 = val_metrics["macro_f1"]
            epochs_no_improve = 0
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "display_name": model_name,
                    "timm_name": timm_name,
                    "backend_source": "torchvision" if model_name in {"shufflenet_v2_x1_0", "squeezenet1_1"} else "timm",
                    "pretrained": pretrained,
                    "num_classes": NUM_CLASSES,
                    "img_size": IMG_SIZE,
                    "class_names": CLASS_NAMES,
                    "augmentation_key": augmentation_key,
                    "augmentation_name": AUTO_AUGMENT_POLICIES[augmentation_key],
                },
                best_path,
            )
            print(f"  --> Saved best checkpoint: val loss {best_val_loss:.4f}, macro F1 {best_val_f1:.4f}")
        else:
            epochs_no_improve += 1
            print(f"  --> No improvement ({epochs_no_improve}/{PATIENCE})")

        if epochs_no_improve >= PATIENCE:
            print("  --> Early stopping triggered.")
            break

    train_time = time.time() - train_start
    checkpoint = torch.load(best_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])

    test_metrics = predict_pytorch(model, test_loader, criterion=None, timed=True)
    inf_time = test_metrics["elapsed"]
    params_m = count_params(model)

    result = {
        "Augmentation": AUTO_AUGMENT_POLICIES[augmentation_key],
        "Augmentation Key": augmentation_key,
        "Model": model_name,
        "Backend Name": timm_name,
        "Parameters (M)": round(params_m, 2),
        "Training Time (s)": round(train_time, 1),
        "Val F1-Score": round(best_val_f1, 4),
        "Best Val Loss": round(best_val_loss, 4),
        "Test Accuracy": round(test_metrics["accuracy"], 4),
        "Test F1-Score": round(test_metrics["macro_f1"], 4),
        "Cohen Kappa": round(test_metrics["cohen_kappa"], 4),
        "Inference Time (s)": round(inf_time, 2),
        "FPS": round(len(test_df) / inf_time, 1),
        "Latency (ms)": round((inf_time / len(test_df)) * 1000, 2),
        "Checkpoint Path": str(best_path),
        "Export Format": "",
        "Export Path": "",
        "Export Note": "",
    }

    del model, optimizer, scheduler
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    return result



## 5. Auto-Augmentation benchmark loop

Runs every TIMM/Torchvision classifier once per augmentation policy. Partial CSV/JSON files are saved after every model.


YOLO classification models are intentionally excluded from this benchmark because they use a separate augmentation pipeline in the Ultralytics trainer.


## 6. Combined results


In [ ]:
comparison_results = []

for augmentation_key, augmentation_name in AUTO_AUGMENT_POLICIES.items():
    print("\n" + "#" * 90)
    print(f"Starting augmentation benchmark: {augmentation_name}")
    print("#" * 90)

    for model_name in ALL_MODELS:
        try:
            reset_random_state(SEED)
            train_loader = make_train_loader(augmentation_key)
            result = train_timm_model(model_name, augmentation_key, train_loader)
            comparison_results.append(result)
            print("Recorded result:")
            display(pd.DataFrame([result]))
        except Exception as exc:
            print(f"ERROR while running {augmentation_name} / {model_name}: {type(exc).__name__}: {exc}")
            comparison_results.append(
                {
                    "Augmentation": augmentation_name,
                    "Augmentation Key": augmentation_key,
                    "Model": model_name,
                    "Backend Name": "",
                    "Parameters (M)": np.nan,
                    "Training Time (s)": np.nan,
                    "Val F1-Score": np.nan,
                    "Best Val Loss": np.nan,
                    "Test Accuracy": np.nan,
                    "Test F1-Score": np.nan,
                    "Cohen Kappa": np.nan,
                    "Inference Time (s)": np.nan,
                    "FPS": np.nan,
                    "Latency (ms)": np.nan,
                    "Checkpoint Path": "",
                    "Export Format": "not_run",
                    "Export Path": "",
                    "Export Note": f"Run failed: {type(exc).__name__}: {exc}",
                }
            )

        partial_df = pd.DataFrame(comparison_results)
        partial_df.to_csv(REPORT_DIR / "autoaugment_model_comparison_partial.csv", index=False)
        partial_df.to_json(REPORT_DIR / "autoaugment_model_comparison_partial.json", orient="records", indent=2)


In [ ]:
df_summary = pd.DataFrame(comparison_results)
metric_columns = [
    "Augmentation",
    "Model",
    "Parameters (M)",
    "Training Time (s)",
    "Val F1-Score",
    "Best Val Loss",
    "Test Accuracy",
    "Test F1-Score",
    "Cohen Kappa",
    "Inference Time (s)",
    "FPS",
    "Latency (ms)",
]

if not df_summary.empty:
    df_summary = df_summary.sort_values(by="Test F1-Score", ascending=False, na_position="last").reset_index(drop=True)
    print("\n" + "=" * 90)
    print("AUTO-AUGMENTATION MODEL PERFORMANCE SUMMARY: SHRIMP DISEASE CLASSIFICATION")
    print("=" * 90)
    display(df_summary[metric_columns])

    summary_csv = REPORT_DIR / "autoaugment_model_comparison_summary.csv"
    summary_json = REPORT_DIR / "autoaugment_model_comparison_summary.json"
    metrics_csv = REPORT_DIR / "autoaugment_model_comparison_metrics_table.csv"
    df_summary.to_csv(summary_csv, index=False)
    df_summary.to_json(summary_json, orient="records", indent=2)
    df_summary[metric_columns].to_csv(metrics_csv, index=False)
    print(f"Saved full summary CSV: {summary_csv}")
    print(f"Saved full summary JSON: {summary_json}")
    print(f"Saved metrics-only CSV: {metrics_csv}")

    best_by_aug = (
        df_summary.sort_values(by="Test F1-Score", ascending=False, na_position="last")
        .groupby("Augmentation", as_index=False)
        .head(1)
        .reset_index(drop=True)
    )
    print("\nBest model per augmentation policy:")
    display(best_by_aug[metric_columns])
else:
    print("No model results were produced.")
